<a href="https://colab.research.google.com/github/raissatsp-cyber/mapeamento-mangue-curimatau/blob/main/master.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

PASSO 1 - Instalação e Importação de Bibliotecas:

Instala os pacotes necessários e importa as bibliotecas para manipulação de vetores, imagens orbitais e integração com o Google Drive.

In [104]:
# 1. Instalação dos pacotes necessários
!pip install geemap geopandas -q

# 2. Importação das bibliotecas principais
import ee
import geemap
import geopandas as gpd
import pandas as pd

PASSO 2 - Montagem do Google Drive:

Conecta o ambiente do Google Colab ao seu Drive para que o código consiga ler o arquivo vetorizado .gpkg e salvar os arquivos finais.

In [105]:
# Montar o Google Drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


PASSO 3 - Autenticação e Inicialização do Google Earth Engine:

Autentica e inicializa a API do Earth Engine utilizando o ID do seu projeto no Google Cloud.

In [106]:
import ee

# 1. Autentica a sessão
ee.Authenticate()

# 2. Inicializa passando o ID do seu projeto no Google Cloud
ee.Initialize(project='mangue-canguaretama')

PASSO 4 - Carregamento da ROI e Seleção das 10 Melhores Imagens

Lê a ROI a partir do GeoPackage guardado no Google Drive e seleciona automaticamente as 10 melhores cenas ópticas (Landsat) e as 10 melhores cenas de radar (Sentinel-1) durante o período seco (Setembro a Janeiro).

In [107]:
# --- PASSO 4 (CORRIGIDO): Padronização de bandas e Seleção das 10 Melhores Imagens ---

# 1. Função para padronizar bandas do Landsat 5 e 7 para a nomenclatura do Landsat 8/9
def renomear_bandas_l57(image):
    bandas_antigas = ['SR_B1', 'SR_B2', 'SR_B3', 'SR_B4', 'SR_B5', 'SR_B7', 'QA_PIXEL']
    bandas_novas   = ['SR_B2', 'SR_B3', 'SR_B4', 'SR_B5', 'SR_B6', 'SR_B7', 'QA_PIXEL']
    return image.select(bandas_antigas, bandas_novas)

# 2. Função para selecionar as bandas equivalentes no Landsat 8 e 9
def selecionar_bandas_l89(image):
    bandas_l89 = ['SR_B2', 'SR_B3', 'SR_B4', 'SR_B5', 'SR_B6', 'SR_B7', 'QA_PIXEL']
    return image.select(bandas_l89)

# 3. Carregar e padronizar todas as coleções
col_l5 = ee.ImageCollection('LANDSAT/LT05/C02/T1_L2').map(mascarar_nuvens_landsat).map(renomear_bandas_l57)
col_l7 = ee.ImageCollection('LANDSAT/LE07/C02/T1_L2').map(mascarar_nuvens_landsat).map(renomear_bandas_l57)
col_l8 = ee.ImageCollection('LANDSAT/LC08/C02/T1_L2').map(mascarar_nuvens_landsat).map(selecionar_bandas_l89)
col_l9 = ee.ImageCollection('LANDSAT/LC09/C02/T1_L2').map(mascarar_nuvens_landsat).map(selecionar_bandas_l89)

# 4. Unificar as coleções (agora perfeitamente compatíveis)
colecao_optica_unificada = col_l5.merge(col_l7).merge(col_l8).merge(col_l9)

# 5. Filtrar as 10 melhores imagens ópticas
melhores_10_opticas = (colecao_optica_unificada
    .filterBounds(aoi)
    .filterDate('1990-01-01', '2025-12-31')
    .filter(ee.Filter.calendarRange(9, 1, 'month'))  # Período seco
    .filter(ee.Filter.lt('CLOUD_COVER', 15))
    .sort('CLOUD_COVER', True)
    .limit(10))

# 6. Filtrar as 10 melhores imagens de radar (Sentinel-1)
colecao_radar = (ee.ImageCollection('COPERNICUS/S1_GRD')
    .filterBounds(aoi)
    .filterDate('2014-01-01', '2025-12-31')
    .filter(ee.Filter.calendarRange(9, 1, 'month'))
    .filter(ee.Filter.eq('instrumentMode', 'IW'))
    .filter(ee.Filter.listContains('transmitterReceiverPolarisation', 'VV'))
    .filter(ee.Filter.listContains('transmitterReceiverPolarisation', 'VH'))
    .filter(ee.Filter.eq('orbitProperties_pass', 'DESCENDING')))

melhores_10_radar = colecao_radar.sort('angle', True).limit(10)

print("✅ Passo 4 atualizado com padronização de bandas!")

✅ Passo 4 atualizado com padronização de bandas!


Passo 5: Inspeção e Visualização Dinâmica das Cenas

Converte as coleções em listas (.toList()) para listar as datas de passagem exatas de cada satélite e inspecionar as imagens.

In [108]:
# Converter coleções para listas de imagens no GEE
lista_opticas = melhores_10_opticas.toList(10)
lista_radar = melhores_10_radar.toList(10)

print("--- 📅 DATAS DAS 10 MELHORES IMAGENS ÓPTICAS ---")
for i in range(10):
    img = ee.Image(lista_opticas.get(i))
    data = img.date().format('YYYY-MM-dd').getInfo()
    print(f"Óptica {i+1}: Passagem em {data}")

print("\n--- 📅 DATAS DAS 10 MELHORES IMAGENS DE RADAR ---")
for i in range(10):
    img = ee.Image(lista_radar.get(i))
    data = img.date().format('YYYY-MM-dd').getInfo()
    print(f"Radar {i+1}: Passagem em {data}")

# Inicializar o mapa interativo
Map = geemap.Map()
Map.centerObject(aoi, 12)

# Adicionar a ROI no mapa
Map.addLayer(aoi, {'color': 'red', 'fillColor': '00000000'}, 'Limite ROI')
Map

--- 📅 DATAS DAS 10 MELHORES IMAGENS ÓPTICAS ---
Óptica 1: Passagem em 2023-10-10
Óptica 2: Passagem em 2025-11-18
Óptica 3: Passagem em 2024-01-09
Óptica 4: Passagem em 1998-12-10
Óptica 5: Passagem em 2022-12-11
Óptica 6: Passagem em 2023-11-06
Óptica 7: Passagem em 2021-10-30
Óptica 8: Passagem em 2010-09-06
Óptica 9: Passagem em 2002-09-08
Óptica 10: Passagem em 2019-10-09

--- 📅 DATAS DAS 10 MELHORES IMAGENS DE RADAR ---
Radar 1: Passagem em 2016-01-17
Radar 2: Passagem em 2016-09-25
Radar 3: Passagem em 2016-09-25
Radar 4: Passagem em 2016-10-07
Radar 5: Passagem em 2016-10-07
Radar 6: Passagem em 2016-10-19
Radar 7: Passagem em 2016-10-19
Radar 8: Passagem em 2016-10-31
Radar 9: Passagem em 2016-10-31
Radar 10: Passagem em 2016-11-12


Map(center=[-6.339385634855069, -35.102036802338965], controls=(WidgetControl(options=['position', 'transparen…

Passo 6: Carregamento das Amostras de Treinamento

Lê a camada de pontos/polígonos de amostra contida no mesmo .gpkg (ou em outro arquivo de validação) para treinar os algoritmos de classificação.

In [109]:
# Rode esta célula para listar os arquivos .gpkg presentes
#no seu ambiente Colab ou Google Drive:

import os
import glob

# 1. Procurar arquivos .gpkg na pasta local do Colab
arquivos_locais = glob.glob("*.gpkg")

# 2. Procurar arquivos .gpkg no Google Drive (se estiver montado)
arquivos_drive = []
if os.path.exists('/content/drive/MyDrive'):
    # Busca arquivos .gpkg no Drive (limite de profundidade para ser rápido)
    for root, dirs, files in os.walk('/content/drive/MyDrive'):
        for file in files:
            if file.endswith('.gpkg'):
                arquivos_drive.append(os.path.join(root, file))

print("📂 --- ARQUIVOS ENCONTRADOS ---")
print("No ambiente local do Colab:", arquivos_locais)
print("No seu Google Drive:", arquivos_drive)

📂 --- ARQUIVOS ENCONTRADOS ---
No ambiente local do Colab: []
No seu Google Drive: ['/content/drive/MyDrive/roi_area_de_estudo.gpkg', '/content/drive/MyDrive/MESTRADO/roi_area_de_estudo.gpkg', '/content/drive/MyDrive/MESTRADO_private/roi_area_de_estudo.gpkg', '/content/drive/MyDrive/MESTRADO_private/amostras_treinamento.gpkg', '/content/drive/MyDrive/aula_pratica/ugrh_aula.gpkg']


In [110]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


Passos 7 e 8: Processamento de Índices, Treinamento e Classificação em Lote

Calcula os índices espectrais/texturais ($NDVI$, $MNDWI$ para óptica e $VH/VV$ para radar), treina dois classificadores Random Forest independentes e aplica o mapeamento via .map() sobre todas as 20 imagens de forma automatizada.

In [111]:
# --- PASSOS 7 E 8: Processamento de Índices, Treinamento e Classificação em Lote ---

def adicionar_indices_opticos(img):
    ndvi = img.normalizedDifference(['SR_B5', 'SR_B4']).rename('NDVI')
    mndwi = img.normalizedDifference(['SR_B3', 'SR_B6']).rename('MNDWI')
    return img.addBands([ndvi, mndwi])

def adicionar_indices_radar(img):
    ratio = img.select('VH').divide(img.select('VV')).rename('VH_VV_ratio')
    return img.addBands(ratio)

col_optica_pronta = melhores_10_opticas.map(adicionar_indices_opticos)
col_radar_pronta = melhores_10_radar.map(adicionar_indices_radar)

bandas_opticas = ['SR_B2', 'SR_B3', 'SR_B4', 'SR_B5', 'SR_B6', 'SR_B7', 'NDVI', 'MNDWI']
bandas_radar = ['VV', 'VH', 'VH_VV_ratio']

# Amostragem dos pixels usando a propriedade 'id_classe'
amostras_optica = col_optica_pronta.select(bandas_opticas).median().sampleRegions(
    collection=amostras_ee, properties=['id_classe'], scale=30, geometries=True
)
amostras_radar = col_radar_pronta.select(bandas_radar).median().sampleRegions(
    collection=amostras_ee, properties=['id_classe'], scale=30, geometries=True
)

# Treinamento com a coluna numérica 'id_classe'
rf_optica = ee.Classifier.smileRandomForest(100).train(
    features=amostras_optica, classProperty='id_classe', inputProperties=bandas_opticas
)
rf_radar = ee.Classifier.smileRandomForest(100).train(
    features=amostras_radar, classProperty='id_classe', inputProperties=bandas_radar
)

# Classificação em lote
classificadas_opticas = col_optica_pronta.map(
    lambda img: img.select(bandas_opticas).classify(rf_optica).set('system:time_start', img.get('system:time_start'))
)
classificadas_radar = col_radar_pronta.map(
    lambda img: img.select(bandas_radar).classify(rf_radar).set('system:time_start', img.get('system:time_start'))
)

print("✅ Passos 7 e 8 atualizados!")

✅ Passos 7 e 8 atualizados!


Passo 9: Avaliação de Acurácia dos Modelos

Avalia a matriz de confusão, acurácia global e índice Kappa de cada modelo antes de exportar os rasters finais.

In [112]:
# --- PASSO 9: Avaliação de Acurácia dos Modelos ---

validacao_optica = amostras_optica.classify(rf_optica)
matriz_optica = validacao_optica.errorMatrix('id_classe', 'classification')

validacao_radar = amostras_radar.classify(rf_radar)
matriz_radar = validacao_radar.errorMatrix('id_classe', 'classification')

print("📊 --- RESULTADOS DE ACURÁCIA DO MODELO ---")
print(f"Óptica | Acurácia Global: {matriz_optica.accuracy().getInfo()*100:.2f}% | Kappa: {matriz_optica.kappa().getInfo():.4f}")
print(f"Radar  | Acurácia Global: {matriz_radar.accuracy().getInfo()*100:.2f}% | Kappa: {matriz_radar.kappa().getInfo():.4f}")

📊 --- RESULTADOS DE ACURÁCIA DO MODELO ---
Óptica | Acurácia Global: 99.80% | Kappa: 0.9975
Radar  | Acurácia Global: 99.37% | Kappa: 0.9920


Passos 10 e 11: Exportação Automatizada dos 20 Rasters para o Google Drive

Dispara as tarefas de exportação em lote para o Google Drive. Cada arquivo .tif é nomeado dinamicamente com o sensor e a data exata em que o satélite passou sobre a área de estudo (ex.: classificacao_optica_2018_11_05.tif).

In [113]:
# Função automatizada para exportar uma coleção inteira de classificações
def exportar_colecao_para_drive(colecao_classificada, tipo_sensor):
    lista_imgs = colecao_classificada.toList(colecao_classificada.size())
    qtd = colecao_classificada.size().getInfo()

    print(f"\n🚀 Disparando tarefas de exportação: {tipo_sensor.upper()} ({qtd} arquivos)")

    for i in range(qtd):
        img = ee.Image(lista_imgs.get(i))

        # Recupera a data exata da imagem no formato AAAA_MM_DD
        data_exata = img.date().format('YYYY_MM_dd').getInfo()

        # Formata o nome do arquivo GeoTIFF
        nome_arquivo = f"classificacao_{tipo_sensor}_{data_exata}"

        # Cria a tarefa de exportação para o Google Drive
        task = ee.batch.Export.image.toDrive(
            image=img.toInt8().clip(aoi),
            description=nome_arquivo,
            folder=f'GEE_Resultados_{tipo_sensor.capitalize()}', # Cria subpasta no Drive
            fileNamePrefix=nome_arquivo,
            scale=30,
            region=aoi,
            fileFormat='GeoTIFF',
            maxPixels=1e9
        )
        task.start()
        print(f" -> Tarefa iniciada: {nome_arquivo}.tif")

# Executar a exportação das 10 imagens ópticas e das 10 de radar
exportar_colecao_para_drive(classificadas_opticas, 'optica')
exportar_colecao_para_drive(classificadas_radar, 'radar')

print("\n🎉 Todas as 20 exportações foram enviadas para a aba 'Tasks' do Earth Engine e serão salvas no seu Google Drive!")


🚀 Disparando tarefas de exportação: OPTICA (10 arquivos)
 -> Tarefa iniciada: classificacao_optica_2023_10_10.tif
 -> Tarefa iniciada: classificacao_optica_2025_11_18.tif
 -> Tarefa iniciada: classificacao_optica_2024_01_09.tif
 -> Tarefa iniciada: classificacao_optica_1998_12_10.tif
 -> Tarefa iniciada: classificacao_optica_2022_12_11.tif
 -> Tarefa iniciada: classificacao_optica_2023_11_06.tif
 -> Tarefa iniciada: classificacao_optica_2021_10_30.tif
 -> Tarefa iniciada: classificacao_optica_2010_09_06.tif
 -> Tarefa iniciada: classificacao_optica_2002_09_08.tif
 -> Tarefa iniciada: classificacao_optica_2019_10_09.tif

🚀 Disparando tarefas de exportação: RADAR (10 arquivos)
 -> Tarefa iniciada: classificacao_radar_2016_01_17.tif
 -> Tarefa iniciada: classificacao_radar_2016_09_25.tif
 -> Tarefa iniciada: classificacao_radar_2016_09_25.tif
 -> Tarefa iniciada: classificacao_radar_2016_10_07.tif
 -> Tarefa iniciada: classificacao_radar_2016_10_07.tif
 -> Tarefa iniciada: classificacao_

ETAPA DE PÓS-PROCESSAMENTO

1. Filtro Espacial de Maioria (Majority Filter)
- O que faz: Passa uma janela móvel (matriz de $3 \times 3$ ou $5 \times 5$ pixels) sobre a imagem classificada. O valor de cada pixel central é substituído pelo valor da classe que mais se repete ao seu redor.
- Objetivo: Eliminar o ruído clássico do Random Forest (conhecido como efeito "sal e pimenta"), em que pixels isolados de sombra, água ou solo são classificados erroneamente no meio de um dossel contínuo de manguezal.

2. Definição da Área Mínima Mapeável (MMU)
- O que faz: Conta o número de pixels adjacentes contínuos pertencentes à mesma classe (connectedPixelCount). Se o tamanho da mancha for inferior a um limite pré-definido (por exemplo, manchas menores que 5 pixels, o equivalente a $4.500\text{ m}^2$ no Landsat), a área é descartada ou substituída pelo valor da classe dominante ao redor.
- Objetivo: Garante coerência ecológica e cartográfica, eliminando pequenas manchas falsas que não representam feições reais em campo.

3. Mascaramento de Água Bruta e Limite Estuarino
- O que faz: Aplica um corte rigoroso usando a máscara do índice MNDWI (ou da calha d'água permanente) sobre a imagem pós-filtrada.
- Objetivo: Evita que a lâmina d'água principal do rio/estuário seja contabilizada na contagem final de pixels de mangue ou solo.

In [114]:
import ee

# -------------------------------------------------------------------------
# ETAPA 1: FILTRO ESPACIAL DE MAIORIA (MODA)
# -------------------------------------------------------------------------
# Define um núcleo (kernel) de vizinhança de 1.5 pixels de raio (matriz ~ 3x3)
kernel = ee.Kernel.circle(radius=1.5, units='pixels')

# Aplica o redutor de moda para substituir o pixel isolado pela classe dominante ao redor
classificado_filtrado = classificado.focalMode(kernel=kernel)


# -------------------------------------------------------------------------
# ETAPA 2: ÁREA MÍNIMA MAPEÁVEL (MMU) - REMOÇÃO DE MANCHAS PEQUENAS
# -------------------------------------------------------------------------
# Limite mínimo de pixels contínuos para manter a mancha (ex: 5 pixels = ~0.45 ha)
mmu_pixels = 5

# Conta a quantidade de pixels conectados pertencentes à mesma classe
conexoes = classificado_filtrado.connectedPixelCount(maxSize=100, cornerConnected=True)

# Mascara e remove manchas menores que o limite estipulado
classificado_mmu = classificado_filtrado.updateMask(conexoes.gte(mmu_pixels))

# Preenche os "buracos" deixados pela remoção das manchas menores com a moda vizinha
classificado_limpo = classificado_mmu.unmask(classificado_filtrado.focalMode(kernel=kernel))


# -------------------------------------------------------------------------
# ETAPA 3: MÁSCARA FINAL DE ÁGUA E RECORTE DO ESTUÁRIO
# -------------------------------------------------------------------------
# Se você tiver a camada MNDWI calculada, pode re-aplicar a máscara de água
# Exemplo: garantir que a água profunda da calha do rio continue sendo estritamente 'Água'
# classificado_final = classificado_limpo.clip(aoi_ee)

classificado_final = classificado_limpo

print("✅ Pós-processamento concluído com sucesso!")

# -------------------------------------------------------------------------
# VISUALIZAÇÃO COMPARATIVA NO COLAB
# -------------------------------------------------------------------------
import geemap.foliumap as geemap

Map = geemap.Map()
Map.centerObject(aoi_ee, 12)

# Paleta de cores para visualização (ajuste os códigos hexadecimais conforme suas classes)
vis_params = {
    'min': 1,
    'max': 4,
    'palette': ['#0000FF', '#006400', '#D2B48C', '#FF0000'] # Ex: Água, Mangue, Apicum, Carcinicultura
}

Map.addLayer(classificado, vis_params, 'Classificado (Bruto)')
Map.addLayer(classificado_final, vis_params, 'Classificado (Pós-Processado)')
Map.addLayerControl()
Map

NameError: name 'classificado' is not defined